# hf_xet smoke test

`hf_xet` is the Hugging Face Xet-protocol client for fast transfer of large files.
It's a Rust-backed (maturin-built) extension consumed internally by `huggingface_hub` — there is no user-facing Python API.
These cells verify the package imports, its native extension is present, and the public-name surface is intact.

In [ ]:
# Example 1: package metadata + module location
import hf_xet
import os

pkg_file = getattr(hf_xet, '__file__', None)
pkg_path = list(hf_xet.__path__)

print(f"__file__: {pkg_file}")
print(f"__path__: {pkg_path}")
print(f"package dir exists: {os.path.isdir(pkg_path[0])}")

assert pkg_path and os.path.isdir(pkg_path[0]), "hf_xet package path must exist"
assert pkg_file is not None, "hf_xet must have a __file__"

In [ ]:
# Example 2: enumerate public attributes (the Rust ext exposes its API at module level)
import hf_xet

all_attrs = dir(hf_xet)
public_attrs = [a for a in all_attrs if not a.startswith('_')]

print(f"Total attributes:  {len(all_attrs)}")
print(f"Public attributes: {len(public_attrs)}")
print(f"Public surface: {public_attrs[:20]}")

# Sanity: a Rust pyo3 module always has at least __name__ and __doc__
assert '__name__' in all_attrs
assert hf_xet.__name__ == 'hf_xet'

In [ ]:
# Example 3: confirm the native Rust extension shared object loaded
import hf_xet
import os
import glob

pkg_dir = list(hf_xet.__path__)[0]
so_files = glob.glob(os.path.join(pkg_dir, '*.so')) + glob.glob(os.path.join(pkg_dir, '*.pyd'))

print(f"Native artifacts in {pkg_dir}:")
for so in so_files:
    size = os.path.getsize(so)
    print(f"  {os.path.basename(so)}  ({size:,} bytes)")

# hf_xet is a maturin/pyo3 build — there must be at least one native shared object
assert so_files, "hf_xet must ship a native shared object (built via maturin)"

In [ ]:
# Example 4: check distribution metadata via importlib.metadata
from importlib import metadata

dist = metadata.distribution('hf_xet')
print(f"Name:    {dist.metadata['Name']}")
print(f"Version: {dist.version}")
summary = dist.metadata.get('Summary')
if summary:
    print(f"Summary: {summary}")

requires = dist.requires or []
print(f"Declared requirements: {len(requires)}")
for r in requires[:10]:
    print(f"  {r}")

assert dist.metadata['Name'].lower().replace('-', '_') == 'hf_xet'
assert dist.version, "hf_xet must have a version"

In [ ]:
# Example 5: hf_xet is discoverable via importlib
import hf_xet
import importlib.util

spec = importlib.util.find_spec('hf_xet')
print(f"find_spec('hf_xet') -> {spec}")
print(f"  origin:        {spec.origin}")
print(f"  submodule_search_locations: {spec.submodule_search_locations}")

# Confirm it's importable a second time (idempotency check)
import importlib
reloaded = importlib.import_module('hf_xet')
assert reloaded is hf_xet, "re-importing hf_xet must return the same module"

print("\nhf_xet smoke test PASSED")
assert spec is not None and spec.submodule_search_locations is not None